# PCA on MNIST

KNN on 784 pixel columns, then on a few principal components; pictures of the digits in 2D and 3D; choosing the number of components.

In [1]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

## The data

MNIST: 70,000 handwritten digits of 28 x 28 pixels, one column per pixel. The first download takes a minute; after that it is read from a local cache. We keep a random 42,000 images, the size of Kaggle's Digit Recognizer `train.csv`.

In [2]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
idx = np.random.default_rng(0).choice(len(X), 42000, replace=False)
X, y = X[idx], y[idx].astype(int)
X.shape   # (42000, 784)

(42000, 784)

In [3]:
# one row back into a 28 x 28 picture
px.imshow(X[0].reshape(28, 28), color_continuous_scale="gray_r", title=f"label: {y[0]}")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.shape, X_test.shape

((33600, 784), (8400, 784))

## KNN on all 784 columns

In [5]:
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

start = time.perf_counter()
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc = accuracy_score(y_test, knn.predict(X_test_s))
print(f"784 columns: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

784 columns: accuracy 0.940, 13.4 s


## KNN on 100 principal components

In [6]:
start = time.perf_counter()
pca = PCA(n_components=100).fit(X_train_s)          # learn the components from the training set only
Z_train, Z_test = pca.transform(X_train_s), pca.transform(X_test_s)
knn = KNeighborsClassifier(n_neighbors=5).fit(Z_train, y_train)
acc = accuracy_score(y_test, knn.predict(Z_test))
print(f"100 components: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

100 components: accuracy 0.954, 52.9 s


## Accuracy as k grows (a few values; each takes a few seconds)

In [7]:
rows = []
for k in [1, 2, 3, 5, 10, 20, 50, 100]:
    p = PCA(n_components=k).fit(X_train_s)
    pred = KNeighborsClassifier(n_neighbors=5).fit(p.transform(X_train_s), y_train).predict(p.transform(X_test_s))
    rows.append({"k": k, "accuracy": accuracy_score(y_test, pred)})
pd.DataFrame(rows)

,k,accuracy
0,1,0.262500
1,2,0.321905
2,3,0.517738
3,5,0.740119
4,10,0.905000
5,20,0.944167
6,50,0.954405
7,100,0.954405


## Pictures: 2D and 3D (drag the 3D plot to turn it; click a digit in the legend to hide it)

In [8]:
p3 = PCA(n_components=3).fit(X_train_s)
Z = p3.transform(X_test_s)
df = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "PC3": Z[:, 2], "digit": y_test.astype(str)})
order = {"digit": [str(i) for i in range(10)]}
px.scatter(df, x="PC1", y="PC2", color="digit", category_orders=order, opacity=0.5)

In [9]:
fig = px.scatter_3d(df, x="PC1", y="PC2", z="PC3", color="digit", category_orders=order, opacity=0.6)
fig.update_traces(marker_size=2)
fig

## What PCA learned

In [10]:
print(p3.explained_variance_)        # the 3 largest eigenvalues
print(p3.explained_variance_ratio_)  # their share of the total variance
print(p3.components_.shape)          # (3, 784): 3 eigenvectors, each with 784 numbers

[40.3838758  28.95979331 27.02398462]
[0.05663769 0.04061561 0.03790067]
(3, 784)


In [11]:
# each component is itself a 28 x 28 picture: which pixels it combines
px.imshow(p3.components_[0].reshape(28, 28), color_continuous_scale="RdBu", title="PC1 as an image")

## Choosing k: keep 90% of the variance

In [12]:
full = PCA().fit(X_train_s)
cum = np.cumsum(full.explained_variance_ratio_)
k90 = int(np.searchsorted(cum, 0.90) + 1)
print("components for 90% of the variance:", k90)
px.line(x=np.arange(1, 785), y=cum, labels={"x": "k", "y": "cumulative explained variance"})

components for 90% of the variance: 228


In [13]:
# scikit-learn can pick k for us: a number between 0 and 1 means "keep this share of the variance"
PCA(n_components=0.90).fit(X_train_s).n_components_

np.int64(228)

## Checks used in the Note: ink and PC1, rare pixels, PCA without scaling

In [14]:
# PC1 against the number of non-zero (inked) pixels in each test image
ink = (X_test > 0).sum(axis=1)
print('correlation of PC1 with ink:', np.corrcoef(Z[:, 0], ink)[0, 1].round(2))   # about 0.80

knn5 = lambda a, b: KNeighborsClassifier(n_neighbors=5).fit(a, y_train).score(b, y_test)
# drop pixels that are non-zero in fewer than 5% of training images, then KNN on the rest (standardised)
keep = (X_train > 0).mean(axis=0) >= 0.05
print(keep.sum(), 'pixels kept, accuracy', knn5(X_train_s[:, keep], X_test_s[:, keep]))   # about 0.959

# PCA on the unscaled pixels
q = PCA(n_components=50).fit(X_train)
print('unscaled PCA, 50 components:', knn5(q.transform(X_train), q.transform(X_test)))   # about 0.973

# Are the later components just noise for KNN? Compare them with a copy whose values are shuffled
# across images: same spread, but no link to the label any more.
A, B = full.transform(X_train_s), full.transform(X_test_s)
rng = np.random.default_rng(1)
def shuffled(M, lo, hi):
    M = M[:, :hi].copy()
    for j in range(lo, hi):
        M[:, j] = rng.permutation(M[:, j])
    return M
print('components 51-784 alone:', knn5(A[:, 50:], B[:, 50:]))                              # about 0.78
print('first 50 + real 51-784:', knn5(A, B))                                              # about 0.940
print('first 50 + shuffled 51-784:', knn5(shuffled(A, 50, 784), shuffled(B, 50, 784)))   # about 0.903
print('components 101-300 alone:', knn5(A[:, 100:300], B[:, 100:300]))                    # about 0.76
print('first 100 + shuffled 101-300:', knn5(shuffled(A, 100, 300), shuffled(B, 100, 300)))  # about 0.933 (real: 0.945)


correlation of PC1 with ink: 0.8


397 pixels kept, accuracy 0.9591666666666666


unscaled PCA, 50 components: 0.9725


components 51-784 alone: 0.7808333333333334


first 50 + real 51-784: 0.9404761904761905


first 50 + shuffled 51-784: 0.9029761904761905


components 101-300 alone: 0.7592857142857142


first 100 + shuffled 101-300: 0.9333333333333333
